## Data Overview
Text

## Key Observations
Text

In [11]:
# Debugging File Path
import sys
print(sys.executable)

/Users/justinlocke/Documents/GitHub/dataunlocked-labs/.venv/bin/python


In [12]:

!pip install pandas
!pip install plotly kaleido
!pip install matplotlib
%pip install python-pptx



Note: you may need to restart the kernel to use updated packages.


In [13]:

import plotly.express as px
import pandas as pd
import kaleido  
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


Source 1) OPTN Portal
The raw dataset is exported in a wide, tab-delimited format intended for human readability rather than analysis. 

Key issues observed:
- Year values are spread across multiple columns
- Numeric values are stored as strings with formatting (e.g., "1 077 213")
- Rows include hierarchical labels and inconsistent structure
- Missing values and irregular spacing are present

This format requires restructuring and cleaning before it can be used for analysis or visualization.

In [14]:
df = pd.read_csv("../data/processed/transplants_treemap.csv")
df.head(10)


,donor_type,organ,transplants
0,Deceased Donor,Kidney,435931
1,Deceased Donor,Liver,227319
2,Living Donor,Kidney,196381
3,Deceased Donor,Heart,102209
4,Deceased Donor,Lung,59258
5,Deceased Donor,Kidney / Pancreas,29548
6,Living Donor,Liver,11147
7,Deceased Donor,Pancreas,9600
8,Deceased Donor,Intestine,3611
9,Deceased Donor,Heart / Lung,1641


In [15]:
df.columns

Index(['donor_type', 'organ', 'transplants'], dtype='str')

## Transformation Summary

The dataset was transformed from a wide, formatted report into a structured tabular format. 

Key steps included:
- Extracting organ categories
- Converting formatted strings to numeric values
- Reshaping year-based columns into a usable structure
- Standardizing field names

The cleaned dataset enables direct comparison across organ categories and supports the categorical visualization.

# 1. Basic structure check

In [16]:

print("Dataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Dataset shape:
(22, 3)

Columns:
['donor_type', 'organ', 'transplants']

Data types:
donor_type       str
organ            str
transplants    int64
dtype: object


# 2. Missing values check

In [17]:
print("\nMissing values:")
print(df.isna().sum())


Missing values:
donor_type     0
organ          0
transplants    0
dtype: int64


In [18]:
# 3. Summary by organ
organ_summary = (
    df.groupby("organ", as_index=False)["transplants"]
    .sum()
    .sort_values("transplants", ascending=False)
)


In [19]:
print("\nTotal transplants by organ:")
display(organ_summary)

# 4. Summary by donor type
donor_type_summary = (
    df.groupby("donor_type", as_index=False)["transplants"]
    .sum()
    .sort_values("transplants", ascending=False)
)


Total transplants by organ:


,organ,transplants
3,Kidney,632312
5,Liver,238466
0,Heart,102252
6,Lung,59511
4,Kidney / Pancreas,29596
7,Pancreas,9624
2,Intestine,3655
1,Heart / Lung,1641
13,VCA - uterus,67
12,VCA - upper limb,38


In [20]:
print("\nTotal transplants by donor type:")
display(donor_type_summary)

# 5. Pivot table: donor type vs organ
donor_organ_pivot = df.pivot_table(
    values="transplants",
    index="donor_type",
    columns="organ",
    aggfunc="sum",
    fill_value=0
)


Total transplants by donor type:


,donor_type,transplants
0,Deceased Donor,869225
1,Living Donor,207988


In [21]:

print("\nPivot table: donor type vs organ:")
display(donor_organ_pivot)



Pivot table: donor type vs organ:


organ,Heart,Heart / Lung,Intestine,Kidney,Kidney / Pancreas,Liver,Lung,Pancreas,VCA - abdominal wall,VCA - external male genitalia,VCA - head and neck,VCA - other genitourinary organ,VCA - upper limb,VCA - uterus
donor_type,,,,,,,,,,,,,,
Deceased Donor,102209,1641,3611,435931,29548,227319,59258,9600,22,2,25,2,38,19
Living Donor,43,0,44,196381,48,11147,253,24,0,0,0,0,0,48


In [22]:

# Final summary
print("""
Summary insights:
- The dataset is structured around donor type, organ category, and transplant counts.
- Organ totals show which transplant categories contribute the largest counts overall.
- Donor type totals compare the relative contribution of each donor source.
- The pivot table makes it easy to compare organ distribution across donor types.
""")


Summary insights:
- The dataset is structured around donor type, organ category, and transplant counts.
- Organ totals show which transplant categories contribute the largest counts overall.
- Donor type totals compare the relative contribution of each donor source.
- The pivot table makes it easy to compare organ distribution across donor types.



In [23]:
# Version 2

fig = px.treemap(
    df,
    path=['organ','donor_type'],
    values='transplants',
    color='organ',
    title="U.S. Transplants by Organ and Donor Type"
)

fig.update_layout(
    margin=dict(t=50, l=25, r=25, b=25)
)

fig.write_image("../outputs/treemap_v2.png")


In [ ]:
# Version 3 ; 4 ; 5 

from pathlib import Path
from IPython.display import HTML, Markdown, display


import plotly.express as px
from IPython.display import Markdown, display


def show_plotly(fig):
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


# Output directory
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Muted Life Flow palette
life_flow_palette = {
    "Kidney": "#6f8f72",
    "Liver": "#7c9a92",
    "Heart": "#9c7a7a",
    "Lung": "#c2a878",
    "Pancreas": "#a8b3a1",
    "Heart / Lung": "#5f7a74",
    "Kidney / Pancreas": "#4f6f55",
    "Intestine": "#b8aa8f",
    "VCA - abdominal wall": "#d6c7aa",
    "VCA - external male genitalia": "#d6c7aa",
    "VCA - head and neck": "#d6c7aa",
    "VCA - other genitourinary organ": "#d6c7aa",
    "VCA - upper limb": "#d6c7aa",
    "VCA - uterus": "#d6c7aa",
}

def style_treemap(fig, title):
    fig.update_traces(
        textinfo="label+value+percent root",
        textfont_size=16,
        marker=dict(line=dict(color="white", width=1.2)),
        hovertemplate="<b>%{label}</b><br>Transplants: %{value:,.0f}<br>Share: %{percentRoot:.1%}<extra></extra>",
    )
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor="left", font=dict(size=22)),
        font=dict(family="Arial, sans-serif", size=14, color="#2f3430"),
        margin=dict(t=60, l=10, r=10, b=10),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    return fig

# Version 3: Organ only
fig_v1 = px.treemap(
    df,
    path=["organ"],
    values="transplants",
    color="organ",
    color_discrete_map=life_flow_palette,
)

fig_v1 = style_treemap(
    fig_v1,
    "U.S. Organ Transplants by Organ Type"
)

fig_v1.write_image(output_dir / "treemap_v3_organ_only.png", scale=2)
show_plotly(fig_v1)

# Version 4: Donor type, then organ
fig_v2 = px.treemap(
    df,
    path=["donor_type", "organ"],
    values="transplants",
    color="organ",
    color_discrete_map=life_flow_palette,
)

fig_v2 = style_treemap(
    fig_v2,
    "U.S. Organ Transplants by Donor Type and Organ"
)

fig_v2.write_image(output_dir / "treemap_v4_donor_then_organ.png", scale=2)
show_plotly(fig_v2)

# Version 5: Organ, then donor type
fig_final = px.treemap(
    df,
    path=["organ", "donor_type"],
    values="transplants",
    color="organ",
    color_discrete_map=life_flow_palette,
)

fig_final = style_treemap(
    fig_final,
    "Kidney Transplants Dominate the U.S. Organ Transplant System"
)

fig_final.write_image(output_dir / "treemap_v5_organ_then_donor.png", scale=2)
show_plotly(fig_final)

display(Markdown("""
- Kidney represents the largest share of transplant volume.
- Living donation is concentrated primarily in kidney transplantation.
- Other organ categories depend largely on deceased donation, reinforcing the supply-constrained nature of the system.
"""))



- Kidney represents the largest share of transplant volume.
- Living donation is concentrated primarily in kidney transplantation.
- Other organ categories depend largely on deceased donation, reinforcing the supply-constrained nature of the system.


In [52]:
# Version 6: Donor type, then organ - Final
df_v2 = df.copy()

donor_soft_palette = {
    "Deceased Donor": "#5f6662",
    "Living Donor": "#c7c3b8",
}

# Keep organs colored by the Life Flow palette while giving parent donor blocks muted colors
df_v2["treemap_color"] = df_v2["organ"]

fig_v2 = px.treemap(
    df_v2,
    path=["donor_type", "organ"],
    values="transplants",
    color="treemap_color",
    color_discrete_map={
        **life_flow_palette,
        **donor_soft_palette,
    },
)

fig_v2.update_traces(
    marker_colors=[
        donor_soft_palette.get(label, life_flow_palette.get(label, "#b8aa8f"))
        for label in fig_v2.data[0].labels
    ]
)

fig_v2 = style_treemap(
    fig_v2,
    "Cumulative U.S. transplants by donor type and organ, 1988–2025"
)



fig_v2.write_image(output_dir / "../outputs/treemap_v6_final.png", scale=2)
fig_final.write_html(output_dir / "../outputs/treemap_v6_final.html")
show_plotly(fig_v2)


In [ ]:

from pathlib import Path

from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor

# Paths
image_path = Path("../outputs/treemap_v6_final.png")
output_dir = Path("../slide")
output_dir.mkdir(parents=True, exist_ok=True)
pptx_path = output_dir / "assignment_03_slide.pptx"

# Colors
bg_color = RGBColor(255, 255, 255)      # #f5f1e8
panel_fill = RGBColor(255, 255, 255)    # #efe7d6
text_color = RGBColor(47, 62, 52)       # #2f3e34

# Create presentation
prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

slide = prs.slides.add_slide(prs.slide_layouts[6])

# Background
slide.background.fill.solid()
slide.background.fill.fore_color.rgb = bg_color

# Treemap image on left
slide.shapes.add_picture(
    str(image_path),
    Inches(0.35),
    Inches(0.45),
    width=Inches(8.05),
    height=Inches(6.35),
)

# Right insight panel
panel = slide.shapes.add_shape(
    MSO_SHAPE.ROUNDED_RECTANGLE,
    Inches(8.7),
    Inches(0.75),
    Inches(4.15),
    Inches(5.9),
)
panel.fill.solid()
panel.fill.fore_color.rgb = panel_fill
panel.line.color.rgb = panel_fill

# Panel text
text_box = slide.shapes.add_textbox(
    Inches(9.05),
    Inches(1.15),
    Inches(3.45),
    Inches(5.0),
)
tf = text_box.text_frame
tf.clear()
tf.word_wrap = True
tf.margin_left = Inches(0.05)
tf.margin_right = Inches(0.05)

# Header
p = tf.paragraphs[0]
p.text = "Key Insights"
p.font.name = "Aptos Display"
p.font.size = Pt(22)
p.font.bold = True
p.font.color.rgb = text_color
p.space_after = Pt(10)



# Bullets
bullets = [
    "The transplant system is heavily concentrated in kidney procedures.",
    "Living donations contributes meaningfully to kidney transplants.",
    "Living donations contribute a small fraction to overall liver transplants.", 
    "Most other organ transplants depend entirely on deceased donors.",
]

for bullet in bullets:
    p = tf.add_paragraph()
    p.text = bullet
    p.level = 0
    p.font.name = "Aptos"
    p.font.size = Pt(14)
    p.font.color.rgb = text_color
    p.space_after = Pt(14)

# Footer
footer_box = slide.shapes.add_textbox(
    Inches(0.45),
    Inches(7.0),
    Inches(8.5),
    Inches(0.3),
)
footer_tf = footer_box.text_frame
footer_tf.clear()

footer_p = footer_tf.paragraphs[0]
footer_p.text = "Source: Organ Procurement and Transplantation Network (OPTN), 2026."
footer_p.font.name = "Aptos"
footer_p.font.size = Pt(9.5)
footer_p.font.color.rgb = text_color

# Save
prs.save(pptx_path)

pptx_path


PosixPath('../slide/assignment_03_slide.pptx')